In [0]:
import pandas as pd
from pyspark.sql import SparkSession

In [0]:
spark = (
    SparkSession.builder
    .getOrCreate()
)

In [0]:
df_bronze = spark.table('db_analytics.bronze').toPandas()
type(df_bronze)

In [0]:

# Defino el inicio de la semana
df_bronze['semana_inicio'] = (
    df_bronze['fecha']
    .dt
    .to_period('W')
    .dt
    .start_time
)

df_bronze.tail(30)

In [0]:
display(df_bronze)

In [0]:
df_gold = (  df_bronze
    .groupby( by=['semana_inicio'] )
    .agg(
        semana_fin = ('fecha', 'max'),
        apertura = ('apertura', 'first'),
        maximo = ('maximo', 'max'),
        minimo = ('minimo', 'min'),
        cierre = ('cierre', 'last'),
        volumen_medio = ('volumen', 'mean')
    )
    .reset_index()
)

df_gold.head()

In [0]:
# Calcular la variacion porcentual

df_gold["var_abs_wow"] = df_gold["cierre"].diff()
df_gold["var_pct_wow"] = df_gold["cierre"].pct_change() * 100
df_gold["vol_pct_wow"] = (df_gold["volumen_medio"].pct_change() * 100)
df_gold["rango"] = (df_gold["maximo"] - df_gold["minimo"]) #rango semanal
df_gold["dif_rango_wow"] = (df_gold["rango"].diff())
df_gold["dif_max_wow"] = (df_gold["maximo"].diff()) 
df_gold["dif_min_wow"] = (df_gold["minimo"].diff())
                          
df_gold.tail()

In [0]:
columnas_2_dec = [
    "apertura",
    "maximo",
    "minimo",
    "cierre",
    "var_abs_wow",
    "var_pct_wow",
    "vol_pct_wow",
    "rango",
    "dif_max_wow",
    "dif_min_wow",
    "dif_rango_wow",
]

df_gold[columnas_2_dec] = df_gold[columnas_2_dec].round(2)


df_gold.tail()

In [0]:
# Convertir un DF Pandas ---> Spark

df_spark = spark.createDataFrame(df_gold)
# Mostrar el DF Spark


# Guardar el DF Spark en un directorio

(
    df_spark.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('db_analytics.precio_semanal_brent')
)

# print(f'Total semanas: {df_spark.count()}')